# 00 -- Data setup and exploration
Audits the G2F 2024/2025 GxE Prediction Competition training data: confirms every
expected file actually contains data (not a failed/HTML download), profiles each
file's shape and columns, and cross-checks environment (year x location) codes
across the trait, meta, soil, weather, and EC files so join keys are understood
before any effect-alone model is built.

No modeling in this notebook. Execution only -- if this needs to run again
after real column names are confirmed against `readme.txt`, extend the checks
below rather than rewriting them.

## Environment setup (Colab or local)

In [1]:
from pathlib import Path
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/g2f_effect_decomposition')
else:
    # Local run (VSCode). Data and results live on Drive; point this at wherever
    # Drive is mounted -- e.g. Google Drive for Desktop on WSL2 is typically
    # under /mnt/g/My Drive/... (adjust drive letter as needed).
    # G2F_BASE_PATH env var overrides this for testing without editing the notebook.
    BASE_PATH = Path(os.environ.get('G2F_BASE_PATH', '/mnt/g/My Drive/g2f_effect_decomposition'))

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")

Mounted at /content/drive
Running on Colab | BASE_PATH = /content/drive/MyDrive/g2f_effect_decomposition


## Imports and config

In [2]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

DATA_DIR = BASE_PATH / 'data' / 'raw' / 'Training_data'

# Filenames as released in the CyVerse Training_data folder.
EXPECTED_FILES = {
    'trait':            '1_Training_Trait_Data_2014_2023.csv',
    'meta':             '2_Training_Meta_Data_2014_2023.csv',
    'soil':             '3_Training_Soil_Data_2015_2023.csv',
    'weather_full':     '4_Training_Weather_Data_2014_2023_full_year.csv',
    'weather_seasons':  '4_Training_Weather_Data_2014_2023_seasons_only.csv',
    'genotype_vcf':     '5_Genotype_Data_All_2014_2025_Hybrids.vcf',
    'genotype_numeric': '5_Genotype_Data_All_2014_2025_Hybrids_numerical.txt',
    'ec':               '6_Training_EC_Data_2014_2023.csv',
    'key_inbreds':      'key_inbreds_G2F_2014-2025.txt',
}

pd.set_option('display.max_columns', 50)

## File integrity check
CyVerse gates downloads behind a reCAPTCHA click-through in its browser app.
A file pulled with a plain HTTP client (rather than a real browser download)
can silently come back as the ~7KB Angular landing page instead of the actual
data. Every file gets checked for that before anything downstream trusts it.

In [3]:
def integrity_status(path: Path) -> str:
    """Returns 'missing', 'suspect_html', or 'ok' for a downloaded file."""
    if not path.exists():
        return 'missing'
    with open(path, 'rb') as f:
        head = f.read(512)
    if b'<!DOCTYPE html' in head or b'<html' in head[:200]:
        return 'suspect_html'
    return 'ok'


status_rows = []
for key, filename in EXPECTED_FILES.items():
    path = DATA_DIR / filename
    status = integrity_status(path)
    size = path.stat().st_size if path.exists() else None
    status_rows.append({'key': key, 'file': filename, 'status': status, 'size_bytes': size})

status_df = pd.DataFrame(status_rows)
print(status_df.to_string(index=False))

bad = status_df[status_df['status'] != 'ok']
if len(bad):
    print(f"\n{len(bad)} file(s) failed the integrity check -- re-download these before"
          " trusting anything profiled below:")
    print(bad['file'].to_string(index=False))
else:
    print("\nAll files passed the integrity check.")

             key                                                file status  size_bytes
           trait                 1_Training_Trait_Data_2014_2023.csv     ok    31590253
            meta                  2_Training_Meta_Data_2014_2023.csv     ok      100054
            soil                  3_Training_Soil_Data_2015_2023.csv     ok       28101
    weather_full     4_Training_Weather_Data_2014_2023_full_year.csv     ok    10460386
 weather_seasons  4_Training_Weather_Data_2014_2023_seasons_only.csv     ok     5493581
    genotype_vcf           5_Genotype_Data_All_2014_2025_Hybrids.vcf     ok    57432657
genotype_numeric 5_Genotype_Data_All_2014_2025_Hybrids_numerical.txt     ok    40765753
              ec                    6_Training_EC_Data_2014_2023.csv     ok     1971244
     key_inbreds                       key_inbreds_G2F_2014-2025.txt     ok      230060

All files passed the integrity check.


## Tabular file profiles
Loads a small sample of each CSV that passed the integrity check: shape,
columns, dtypes, and a head preview. Skips any file that failed the check
above rather than profiling garbage.

In [4]:
TABULAR_KEYS = ['trait', 'meta', 'soil', 'weather_full', 'weather_seasons', 'ec']

tabular_frames: dict[str, pd.DataFrame] = {}

for key in TABULAR_KEYS:
    filename = EXPECTED_FILES[key]
    row = status_df[status_df['key'] == key].iloc[0]
    if row['status'] != 'ok':
        print(f"[skip] {key} ({filename}): {row['status']}")
        continue

    df_full = pd.read_csv(DATA_DIR / filename)
    tabular_frames[key] = df_full

    print(f"=== {key} ({filename}) ===")
    print(f"shape: {df_full.shape}")
    print(f"columns: {list(df_full.columns)}")
    print(df_full.head(3))
    print()

=== trait (1_Training_Trait_Data_2014_2023.csv) ===
shape: (173960, 26)
columns: ['Env', 'Year', 'Field_Location', 'Experiment', 'Replicate', 'Block', 'Plot', 'Range', 'Pass', 'Hybrid', 'Hybrid_orig_name', 'Hybrid_Parent1', 'Hybrid_Parent2', 'Plot_Area_ha', 'Date_Planted', 'Date_Harvested', 'Stand_Count_plants', 'Pollen_DAP_days', 'Silk_DAP_days', 'Plant_Height_cm', 'Ear_Height_cm', 'Root_Lodging_plants', 'Stalk_Lodging_plants', 'Yield_Mg_ha', 'Grain_Moisture', 'Twt_kg_m3']
         Env  Year Field_Location   Experiment  Replicate  Block  Plot  Range  \
0  DEH1_2014  2014           DEH1  G2F_2014_15          1      1     1    1.0   
1  DEH1_2014  2014           DEH1  G2F_2014_15          1      1     2    1.0   
2  DEH1_2014  2014           DEH1  G2F_2014_15          1      1     3    1.0   

   Pass       Hybrid Hybrid_orig_name Hybrid_Parent1 Hybrid_Parent2  \
0   1.0  M0088/LH185      M0088/LH185          M0088          LH185   
1   2.0  M0143/LH185      M0143/LH185          M0143  

In [38]:
print(tabular_frames['trait'].shape)
tabular_frames['trait']

(173960, 26)


,Env,Year,Field_Location,Experiment,Replicate,Block,Plot,Range,Pass,Hybrid,Hybrid_orig_name,Hybrid_Parent1,Hybrid_Parent2,Plot_Area_ha,Date_Planted,Date_Harvested,Stand_Count_plants,Pollen_DAP_days,Silk_DAP_days,Plant_Height_cm,Ear_Height_cm,Root_Lodging_plants,Stalk_Lodging_plants,Yield_Mg_ha,Grain_Moisture,Twt_kg_m3
0,DEH1_2014,2014,DEH1,G2F_2014_15,1,1,1,1.0,1.0,M0088/LH185,M0088/LH185,M0088,LH185,0.000716,5/5/14,9/29/14,56.0,63.0,67.0,213.000000,79.000000,0.0,0.0,5.721725,20.8,706.664693
1,DEH1_2014,2014,DEH1,G2F_2014_15,1,1,2,1.0,2.0,M0143/LH185,M0143/LH185,M0143,LH185,0.000716,5/5/14,9/29/14,54.0,61.0,63.0,286.000000,172.000000,0.0,0.0,11.338246,25.8,693.792841
2,DEH1_2014,2014,DEH1,G2F_2014_15,1,1,3,1.0,3.0,M0003/LH185,M0003/LH185,M0003,LH185,0.000716,5/5/14,9/29/14,60.0,63.0,65.0,239.000000,92.000000,0.0,4.0,6.540810,20.8,698.941582
3,DEH1_2014,2014,DEH1,G2F_2014_15,1,1,4,1.0,4.0,M0035/LH185,M0035/LH185,M0035,LH185,0.000716,5/5/14,9/29/14,59.0,61.0,63.0,242.000000,118.000000,0.0,0.0,10.366857,23.7,711.813434
4,DEH1_2014,2014,DEH1,G2F_2014_15,1,1,5,1.0,5.0,M0052/LH185,M0052/LH185,M0052,LH185,0.000716,5/5/14,9/29/14,58.0,63.0,65.0,211.000000,92.000000,0.0,0.0,10.908814,19.4,743.993065
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173955,WIH3_2023,2023,WIH3,HIP_Hybrid,2,1,1228,18.0,15.0,PHB47/MO17,PHB47/MO17,PHB47,MO17,0.000929,4/26/23,11/14/23,74.0,81.0,82.0,263.333333,135.000000,0.0,0.0,16.979288,21.7,719.536546
173956,WIH3_2023,2023,WIH3,HIP_Hybrid,2,1,1229,18.0,16.0,PHJ40/PHAJ0,PHJ40/PHAJ0,PHJ40,PHAJ0,0.000929,4/26/23,11/14/23,80.0,70.0,70.0,226.666667,105.000000,0.0,0.0,10.395122,18.0,776.172696
173957,WIH3_2023,2023,WIH3,HIP_Hybrid,2,1,1230,18.0,17.0,LH244/PHK76,LH244/PHK76,LH244,PHK76,0.000929,4/26/23,11/14/23,71.0,78.0,80.0,231.666667,110.000000,0.0,0.0,14.129059,21.9,760.726473
173958,WIH3_2023,2023,WIH3,HIP_Hybrid,2,1,1231,18.0,18.0,B73/MO17,B73/MO17,B73,MO17,0.000929,4/26/23,11/14/23,76.0,84.0,100.0,258.333333,145.000000,0.0,0.0,17.255379,23.7,674.485063


In [37]:
print(tabular_frames['meta'].shape)
tabular_frames['meta'].head(5)

(272, 38)


,Year,Env,Experiment_Code,Treatment,City,Farm,Field,Trial_ID (Assigned by collaborator for internal reference),"Soil_Taxonomic_ID and horizon description, if known","Weather_Station_Serial_Number (Last four digits, e.g. m2700s#####)",Weather_Station_Latitude (in decimal numbers NOT DMS),Weather_Station_Longitude (in decimal numbers NOT DMS),Date_weather_station_placed,Date_weather_station_removed,Previous_Crop,Pre-plant_tillage_method(s),In-season_tillage_method(s),Type_of_planter (fluted cone; belt cone; air planter),System_Determining_Moisture,Pounds_Needed_Soil_Moisture,Latitude_of_Field_Corner_#1 (lower left),Longitude_of_Field_Corner_#1 (lower left),Latitude_of_Field_Corner_#2 (lower right),Longitude_of_Field_Corner_#2 (lower right),Latitude_of_Field_Corner_#3 (upper right),Longitude_of_Field_Corner_#3 (upper right),Latitude_of_Field_Corner_#4 (upper left),Longitude_of_Field_Corner_#4 (upper left),Cardinal_Heading_Pass_1,Irrigated,Issue/comment_#1,Issue/comment_#2,Issue/comment_#3,Issue/comment_#4,Issue/comment_#5,Issue/comment_#6,Issue/comment_#7,Issue/comment_#8
0,2014,DEH1_2014,DEH1,NaN,Georgetown,Elbert N. & Ann V. Carvel Research & Education...,27AB,NaN,NaN,9079,38.637405,-75.204048,NaN,NaN,soybean,Conventional,NaN,Air planter,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014,GAH1_2014,GAH1,NaN,Tifton,USDA - Bellflower experimental farm,18,NaN,NaN,8427,31.506544,-83.555016,NaN,NaN,cotton,conventional,NaN,fluted cone,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014,IAH1a_2014,IAH1,NaN,Ames,Worle,NaN,NaN,NaN,9080,41.996530,-93.696188,NaN,NaN,soybean,field cultivator,NaN,Air planter,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Information for IAH1a_2014, IAH1b_2014,and IAH...",NaN
3,2014,IAH1b_2014,IAH1,NaN,Ames,Worle,NaN,NaN,NaN,9080,41.996530,-93.696188,NaN,NaN,soybean,field cultivator,NaN,Air planter,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Information for IAH1a_2014, IAH1b_2014,and IAH...",NaN
4,2014,IAH1c_2014,IAH1,NaN,Ames,Worle,NaN,NaN,NaN,9080,41.996530,-93.696188,NaN,NaN,soybean,field cultivator,NaN,Air planter,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Information for IAH1a_2014, IAH1b_2014,and IAH...",NaN


In [35]:
print(tabular_frames['soil'].shape)
tabular_frames['soil'].head(5)

(186, 36)


,Year,Env,LabID,Date Received,Date Reported,E Depth,1:1 Soil pH,WDRF Buffer pH,1:1 S Salts mmho/cm,Texture No,Organic Matter LOI %,Nitrate-N ppm N,lbs N/A,Potassium ppm K,Sulfate-S ppm S,Calcium ppm Ca,Magnesium ppm Mg,Sodium ppm Na,CEC/Sum of Cations me/100g,%H Sat,%K Sat,%Ca Sat,%Mg Sat,%Na Sat,Mehlich P-III ppm P,% Sand,% Silt,% Clay,Texture,BpH,Zinc ppm Zn,Iron ppm Fe,Manganese ppm Mn,Copper ppm Cu,Boron ppm B,Comments
0,2015,IAH1_2015,UW Soil & Plant Analysis Lab,NaN,5/27/2015,NaN,5.3,NaN,NaN,NaN,3.2,NaN,NaN,118.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34.0,NaN,NaN,NaN,NaN,6.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,IAH3_2015,UW Soil & Plant Analysis Lab,NaN,5/27/2015,NaN,6.5,NaN,NaN,NaN,3.4,NaN,NaN,114.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.0,NaN,NaN,NaN,NaN,6.5,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,IAH4_2015,Soil & Forage Analysis Lab,NaN,12/7/2015,NaN,6.4,NaN,NaN,NaN,4.4,NaN,NaN,95.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42.0,NaN,NaN,NaN,NaN,6.4,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,INH1_2015,UW Soil & Plant Analysis Lab,NaN,6/2/2015,NaN,6.1,NaN,NaN,NaN,2.9,NaN,NaN,86.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.0,NaN,NaN,NaN,NaN,6.5,NaN,NaN,NaN,NaN,NaN,NaN
4,2015,KSH1_2015,UW Soil & Plant Analysis Lab,NaN,5/8/2015,NaN,6.5,NaN,NaN,NaN,2.1,NaN,NaN,145.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,NaN,NaN,NaN,NaN,6.7,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
print(tabular_frames['weather_full'].shape)
tabular_frames['weather_full'].head(5)

(98236, 18)


,Env,Date,RH2M,T2M_MAX,ALLSKY_SFC_SW_DWN,T2MWET,GWETTOP,QV2M,GWETPROF,T2M_MIN,T2MDEW,PS,T2M,GWETROOT,ALLSKY_SFC_PAR_TOT,WS2M,ALLSKY_SFC_SW_DNI,PRECTOTCORR
0,DEH1_2015,20150101,74.81,6.00,10.75,-1.61,0.77,2.81,0.74,-3.39,-3.86,102.08,0.63,0.77,50.83,3.13,26.76,0.00
1,DEH1_2015,20150102,79.81,8.23,5.72,0.59,0.77,3.42,0.74,-1.47,-1.08,102.44,2.26,0.77,28.82,2.02,2.59,0.00
2,DEH1_2015,20150103,92.81,7.77,1.74,3.99,0.78,4.94,0.74,-1.35,3.57,102.70,4.41,0.77,9.40,1.90,1.36,6.70
3,DEH1_2015,20150104,95.00,18.62,4.05,12.99,0.80,9.34,0.75,6.78,12.58,101.20,13.40,0.78,21.84,5.38,2.17,9.98
4,DEH1_2015,20150105,63.38,8.73,10.92,-0.87,0.79,2.87,0.76,-3.14,-4.08,102.41,2.34,0.79,51.96,5.17,27.70,0.00


In [ ]:
print(tabular_frames['weather_seasons'].shape)
tabular_frames['weather_seasons'].head(5)


(51098, 18)


,Env,Date,RH2M,T2M_MAX,ALLSKY_SFC_SW_DWN,T2MWET,GWETTOP,QV2M,GWETPROF,T2M_MIN,T2MDEW,PS,T2M,GWETROOT,ALLSKY_SFC_PAR_TOT,WS2M,ALLSKY_SFC_SW_DNI,PRECTOTCORR
0,DEH1_2015,20150415,69.12,16.03,15.94,9.57,0.88,5.98,0.88,9.23,6.57,102.19,12.56,0.91,83.09,2.70,6.85,0.28
1,DEH1_2015,20150416,75.00,19.06,25.09,10.77,0.87,6.77,0.88,7.72,8.44,102.68,13.10,0.91,126.43,2.54,27.66,0.01
2,DEH1_2015,20150417,88.94,22.12,16.35,15.49,0.87,10.25,0.87,11.17,14.51,101.69,16.48,0.91,87.12,3.07,14.00,3.04
3,DEH1_2015,20150418,79.75,23.44,24.56,16.05,0.87,9.95,0.86,12.42,14.12,101.44,18.00,0.90,126.23,1.41,28.05,0.36
4,DEH1_2015,20150419,84.44,14.35,18.07,11.32,0.87,7.57,0.86,10.57,9.98,101.77,12.65,0.89,94.41,5.13,8.14,8.92


In [33]:
print(tabular_frames['ec'].shape)
tabular_frames['ec'].head(5)

(241, 655)


,Env,HI30_pGerEme,HI30_pEmeEnJ,HI30_pEnJFlo,HI30_pFloFla,HI30_pFlaFlw,HI30_pFlwStG,HI30_pStGEnG,HI30_pEnGMat,HI30_pMatHar,CumHI30_pGerEme,CumHI30_pEmeEnJ,CumHI30_pEnJFlo,CumHI30_pFloFla,CumHI30_pFlaFlw,CumHI30_pFlwStG,CumHI30_pStGEnG,CumHI30_pEnGMat,CumHI30_pMatHar,TT_pGerEme,TT_pEmeEnJ,TT_pEnJFlo,TT_pFloFla,TT_pFlaFlw,TT_pFlwStG,...,SWmm_pFloFla_9,SWmm_pFlaFlw_9,SWmm_pFlwStG_9,SWmm_pStGEnG_9,SWmm_pEnGMat_9,SWmm_pMatHar_9,SWmm_pGerEme_10,SWmm_pEmeEnJ_10,SWmm_pEnJFlo_10,SWmm_pFloFla_10,SWmm_pFlaFlw_10,SWmm_pFlwStG_10,SWmm_pStGEnG_10,SWmm_pEnGMat_10,SWmm_pMatHar_10,LL__1,LL__2,LL__3,LL__4,LL__5,LL__6,LL__7,LL__8,LL__9,LL__10
0,DEH1_2014,1,5,0,23,4,7,26,2,1,1,6,6,29,33,40,66,68,69,11.469278,11.247966,10.538930,15.356142,19.133298,17.778750,...,0.275170,0.268841,0.268670,0.268749,0.268871,0.268878,0.275216,0.355693,0.390527,0.390527,0.390491,0.389423,0.383818,0.380066,0.379987,0.030565,0.030565,0.030565,0.030269,0.069334,0.069334,0.179364,0.179364,0.179364,0.167243
1,GAH1_2014,0,2,0,26,4,7,29,2,1,0,2,2,28,32,39,68,70,71,10.224345,10.978884,9.647537,14.612056,16.433000,17.659286,...,0.254581,0.230066,0.229570,0.229690,0.230126,0.230189,0.386261,0.386261,0.386261,0.386261,0.386010,0.383307,0.371613,0.363717,0.363590,0.022565,0.022565,0.029567,0.029761,0.029761,0.088368,0.088368,0.148904,0.148904,0.148904
2,IAH1c_2014,2,6,1,15,3,3,16,2,1,2,8,9,24,27,30,46,48,49,9.293041,13.442194,12.073785,13.115922,14.493964,13.636890,...,0.192470,0.230789,0.211378,0.207039,0.207109,0.207107,0.150153,0.150291,0.150387,0.196841,0.384380,0.384356,0.378225,0.372698,0.372550,0.175867,0.175867,0.181313,0.181053,0.162927,0.162927,0.162927,0.090938,0.090938,0.090938
3,IAH2_2014,1,6,0,13,3,2,14,0,0,1,7,7,20,23,25,39,39,39,10.395753,13.318299,9.135988,12.842650,14.578384,12.936901,...,0.185089,0.228831,0.211027,0.207014,0.207116,0.207182,0.150061,0.150170,0.150244,0.186369,0.375300,0.384252,0.376043,0.369711,0.369415,0.127380,0.127380,0.127254,0.127035,0.127035,0.127035,0.127035,0.090938,0.090938,0.090938
4,IAH3_2014,0,4,0,18,1,1,15,1,0,0,4,4,22,23,24,39,40,40,11.405717,12.227550,11.469780,13.584012,12.102371,12.998919,...,0.301661,0.347159,0.343788,0.328083,0.318886,0.318501,0.252431,0.252690,0.252888,0.287429,0.393478,0.385996,0.363861,0.349900,0.349395,0.182344,0.182344,0.182344,0.198777,0.198581,0.198581,0.198581,0.153790,0.153790,0.153790


## Missing-value profile
Per column missing-value percentage for each loaded tabular file -- flags
which fields are usable as-is vs. need imputation or exclusion.

In [5]:
for key, df_full in tabular_frames.items():
    miss = (df_full.isna().mean() * 100).round(1)
    miss = miss[miss > 0].sort_values(ascending=False)
    print(f"=== {key}: missing % by column ===")
    print(miss.to_string() if len(miss) else "(no missing values)")
    print()

=== trait: missing % by column ===
Root_Lodging_plants     38.1
Stalk_Lodging_plants    30.4
Silk_DAP_days           24.2
Pollen_DAP_days         22.8
Twt_kg_m3               21.4
Plant_Height_cm         12.8
Ear_Height_cm           11.9
Range                   10.8
Pass                    10.6
Stand_Count_plants       9.9
Grain_Moisture           5.4
Yield_Mg_ha              5.2
Hybrid_Parent1           1.0
Hybrid_Parent2           1.0
Date_Harvested           0.7
Date_Planted             0.2

=== meta: missing % by column ===
Issue/comment_#8                                                      99.6
Issue/comment_#6                                                      99.3
Issue/comment_#5                                                      97.1
Issue/comment_#7                                                      96.3
Issue/comment_#4                                                      92.6
Issue/comment_#3                                                      89.3
Irrigated       

## Environment (year x location) join-key audit
G2F's known misjoin risk is in the environment code used to link trait,
meta, soil, weather, and EC records. This looks for any column whose name
contains 'env' (case-insensitive) in each loaded file, then compares the
sets of values across files -- mismatches here mean the join key isn't as
simple as a direct string match and the readme needs to be checked before
building any loader.

In [6]:
env_value_sets: dict[str, set] = {}

for key, df_full in tabular_frames.items():
    env_cols = [c for c in df_full.columns if 'env' in c.lower()]
    if not env_cols:
        print(f"{key}: no column with 'env' in its name -- columns are {list(df_full.columns)}")
        continue
    col = env_cols[0]
    env_value_sets[key] = set(df_full[col].astype(str).unique())
    print(f"{key}: using column '{col}' ({df_full[col].nunique()} unique values)"
          + (f" -- also found {env_cols[1:]}" if len(env_cols) > 1 else ""))

print()
keys = list(env_value_sets.keys())
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        a, b = keys[i], keys[j]
        overlap = env_value_sets[a] & env_value_sets[b]
        only_a = env_value_sets[a] - env_value_sets[b]
        only_b = env_value_sets[b] - env_value_sets[a]
        print(f"{a} vs {b}: {len(overlap)} shared, {len(only_a)} only in {a}, {len(only_b)} only in {b}")
        if only_a:
            print(f"  sample only in {a}: {sorted(only_a)[:5]}")
        if only_b:
            print(f"  sample only in {b}: {sorted(only_b)[:5]}")

trait: using column 'Env' (272 unique values)
meta: using column 'Env' (272 unique values)
soil: using column 'Env' (186 unique values)
weather_full: using column 'Env' (269 unique values)
weather_seasons: using column 'Env' (269 unique values)
ec: using column 'Env' (241 unique values)

trait vs meta: 272 shared, 0 only in trait, 0 only in meta
trait vs soil: 186 shared, 86 only in trait, 0 only in soil
  sample only in trait: ['ARH1_2017', 'ARH1_2018', 'ARH2_2017', 'ARH2_2018', 'COH1_2021']
trait vs weather_full: 269 shared, 3 only in trait, 0 only in weather_full
  sample only in trait: ['TXH2_2015', 'TXH2_2016', 'TXH2_2017']
trait vs weather_seasons: 269 shared, 3 only in trait, 0 only in weather_seasons
  sample only in trait: ['TXH2_2015', 'TXH2_2016', 'TXH2_2017']
trait vs ec: 241 shared, 31 only in trait, 0 only in ec
  sample only in trait: ['GEH1_2019', 'GEH1_2020', 'GEH1_2021', 'GEH1_2022', 'IAH1_2016']
meta vs soil: 186 shared, 86 only in meta, 0 only in soil
  sample only 

## Genotype data
Profiles both genotype formats without loading the full VCF body into
memory -- header/sample/contig info only via `cyvcf2`, plus a shape check
on the numerical marker matrix and a preview of the key-inbreds list.

In [8]:
!pip install cyvcf2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 43.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.9 MB/s eta 0:00:00


In [9]:
geno_vcf_status = status_df[status_df['key'] == 'genotype_vcf'].iloc[0]['status']

if geno_vcf_status == 'ok':
    from cyvcf2 import VCF

    vcf_path = DATA_DIR / EXPECTED_FILES['genotype_vcf']
    vcf = VCF(str(vcf_path))
    print(f"VCF samples: {len(vcf.samples)}")
    print(f"VCF sample preview: {vcf.samples[:5]}")

    n_variants = 0
    contigs = set()
    for i, variant in enumerate(vcf):
        contigs.add(variant.CHROM)
        n_variants += 1
        if i >= 9999:  # cap the scan -- full-file variant count can be done separately if needed
            print("(stopped after 10,000 variants -- re-run without the cap for an exact count)")
            break
    print(f"variants scanned: {n_variants}")
    print(f"contigs seen: {sorted(contigs)}")
else:
    print(f"[skip] genotype_vcf: {geno_vcf_status}")

VCF samples: 5899
VCF sample preview: ['01CSI6/LH287', '01DIB2/LH287', '01DIB2/PHP02', '2369/DK3IIH6', '2369/LH123HT']
variants scanned: 2425
contigs seen: ['1', '10', '2', '3', '4', '5', '6', '7', '8', '9']


In [40]:
vcf.samples

['01CSI6/LH287',
 '01DIB2/LH287',
 '01DIB2/PHP02',
 '2369/DK3IIH6',
 '2369/LH123HT',
 '2369/PHN82',
 '2369/PHZ51',
 '2FACC/DK3IIH6',
 '2FACC/LH287',
 '2FACC/PHP02',
 '4N506/DK3IIH6',
 '6F629/DK3IIH6',
 '6F629/LH287',
 '6F629/PHP02',
 '740/PB80',
 '740/PHB47',
 'A632/DK3IIH6',
 'A634/DK3IIH6',
 'A634/LH82',
 'A634/PHZ51',
 'A635/DK3IIH6',
 'A635/LH82',
 'A635/PHZ51',
 'A641/LH82',
 'A641/PHZ51',
 'A672/LH82',
 'A672/PHZ51',
 'A679/DK3IIH6',
 'A679/LH82',
 'A679/PHZ51',
 'A680/DK3IIH6',
 'A680/LH82',
 'A680/PHZ51',
 'AH83/DK3IIH6',
 'B103/LH82',
 'B103/PHB47',
 'B103/PHZ51',
 'B104/DK3IIH6',
 'B104/LH82',
 'B104/PHZ51',
 'B105/DK3IIH6',
 'B105/LH82',
 'B105/PHZ51',
 'B106/LH82',
 'B106/PHZ51',
 'B109/DK3IIH6',
 'B109/LH82',
 'B109/PHZ51',
 'B110/DK3IIH6',
 'B111/DK3IIH6',
 'B111/LH82',
 'B111/PHZ51',
 'B118/PHB47',
 'B119/DK3IIH6',
 'B119/LH82',
 'B119/PHZ51',
 'B14/DK3IIH6',
 'B14A/C103',
 'B14A/DK3IIH6',
 'B14A/H95',
 'B14A/LH82',
 'B14A/MO17',
 'B14A/OH43',
 'B2/DK3IIH6',
 'B37/C103',

In [10]:
geno_num_status = status_df[status_df['key'] == 'genotype_numeric'].iloc[0]['status']

if geno_num_status == 'ok':
    geno_num_path = DATA_DIR / EXPECTED_FILES['genotype_numeric']

    # First line is a format tag (e.g. '<Numeric>'), not part of the
    # tab-separated header -- sep=None sniffing gets fooled by it since the
    # tag line itself has no delimiter to detect. Skip it and read explicitly.
    with open(geno_num_path) as f:
        tag_line = f.readline().strip()
    print(f"format tag line: {tag_line!r}")

    geno_num = pd.read_csv(geno_num_path, sep='\t', skiprows=1, index_col=0)
    geno_num.index.name = 'Hybrid'
    print(f"numerical genotype matrix shape: {geno_num.shape}  (hybrids x markers)")
    print(f"hybrid preview: {list(geno_num.index[:5])}")
    print(f"marker preview: {list(geno_num.columns[:5])}")
    print(geno_num.iloc[:5, :5])
    print()
    print(f"value counts (first marker column): {geno_num.iloc[:, 0].value_counts(dropna=False).to_dict()}")
else:
    print(f"[skip] genotype_numeric: {geno_num_status}")

format tag line: '<Numeric>'
numerical genotype matrix shape: (5899, 2425)  (hybrids x markers)
hybrid preview: ['01CSI6/LH287', '01DIB2/LH287', '01DIB2/PHP02', '2369/DK3IIH6', '2369/LH123HT']
marker preview: ['S1_1007742', 'S1_1020677', 'S1_2018002', 'S1_2101934', 'S1_2275970']
              S1_1007742  S1_1020677  S1_2018002  S1_2101934  S1_2275970
Hybrid                                                                  
01CSI6/LH287         0.5         1.0         1.0         1.0         1.0
01DIB2/LH287         1.0         1.0         1.0         1.0         1.0
01DIB2/PHP02         1.0         0.5         0.5         NaN         1.0
2369/DK3IIH6         0.5         1.0         0.5         1.0         1.0
2369/LH123HT         0.5         0.5         1.0         0.5         0.5

value counts (first marker column): {1.0: 2968, 0.5: 2566, 0.0: 346, nan: 19}


In [41]:
geno_num

,S1_1007742,S1_1020677,S1_2018002,S1_2101934,S1_2275970,S1_2800964,S1_2811950,S1_2888631,S1_3023078,S1_3027593,S1_3113549,S1_3165324,S1_3224170,S1_3370337,S1_3371362,S1_4082864,S1_4083443,S1_4083493,S1_4091412,S1_4786541,S1_4860666,S1_5050985,S1_5051275,S1_5840281,S1_6247673,...,S10_146904768,S10_146916853,S10_146928552,S10_147173783,S10_147282479,S10_147282490,S10_147352179,S10_148272189,S10_148510710,S10_148511076,S10_149259041,S10_149298931,S10_149394364,S10_149474479,S10_149508570,S10_149647241,S10_149717807,S10_149851103,S10_149865723,S10_150351135,S10_150646162,S10_150711963,S10_150733250,S10_151045975,S10_151157757
Hybrid,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
01CSI6/LH287,0.5,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,1.0,1.0,0.5,0.0,1.0,1.0,1.0,1.0,0.5,1.0,1.0,0.5,0.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.5,0.0,1.0,1.0,0.5,1.0,0.0,1.0,0.0,0.0,0.0,0.5,0.0,1.0
01DIB2/LH287,1.0,1.0,1.0,1.0,1.0,0.5,0.5,1.0,1.0,1.0,1.0,0.5,0.0,1.0,1.0,1.0,0.5,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.5,...,1.0,0.5,1.0,0.5,1.0,1.0,0.5,0.5,0.5,0.5,0.5,0.5,0.5,1.0,1.0,0.5,1.0,0.5,0.5,0.5,0.5,0.5,0.5,0.5,1.0
01DIB2/PHP02,1.0,0.5,0.5,NaN,1.0,0.0,0.0,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,1.0,0.0,1.0,0.5,1.0,0.5,0.5,1.0,1.0,0.0,...,1.0,0.5,1.0,0.5,0.5,0.5,1.0,0.5,0.5,0.5,1.0,0.5,0.5,1.0,1.0,1.0,0.5,1.0,0.5,0.5,1.0,1.0,0.5,1.0,1.0
2369/DK3IIH6,0.5,1.0,0.5,1.0,1.0,0.5,0.5,0.5,1.0,0.5,0.5,1.0,1.0,1.0,0.5,0.5,0.5,1.0,0.5,1.0,0.5,1.0,0.5,1.0,0.5,...,1.0,0.5,1.0,0.5,0.5,0.5,1.0,0.5,0.5,0.5,0.5,0.5,1.0,0.5,0.5,1.0,0.5,0.5,0.5,1.0,0.5,0.5,0.5,1.0,1.0
2369/LH123HT,0.5,0.5,1.0,0.5,0.5,1.0,1.0,0.5,1.0,0.5,1.0,0.5,0.5,1.0,0.5,0.5,1.0,1.0,1.0,1.0,0.0,1.0,0.5,1.0,1.0,...,1.0,0.5,1.0,0.5,1.0,1.0,0.5,0.5,0.5,0.5,0.5,0.5,0.5,1.0,1.0,0.5,1.0,0.5,0.5,1.0,0.0,1.0,0.5,1.0,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Z038E0057/PHZ51,0.5,1.0,1.0,1.0,1.0,0.5,1.0,0.5,1.0,0.5,0.5,1.0,0.5,1.0,0.5,0.5,1.0,0.5,0.5,1.0,0.5,0.5,0.5,0.5,0.5,...,1.0,0.5,1.0,0.5,1.0,1.0,0.5,0.5,0.5,0.5,0.5,0.5,1.0,0.5,1.0,1.0,0.5,0.5,0.5,1.0,0.0,1.0,0.0,1.0,1.0
ZS01459/LH287,1.0,1.0,1.0,1.0,1.0,1.0,0.5,1.0,1.0,1.0,0.5,0.5,0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,0.5,0.5,1.0,0.5,1.0,...,1.0,1.0,1.0,1.0,1.0,0.5,0.5,1.0,1.0,1.0,0.5,1.0,0.0,1.0,1.0,0.5,0.5,0.0,1.0,0.5,0.5,0.0,0.5,0.5,0.5
ZS01459/PHP02,1.0,0.5,0.5,NaN,1.0,0.5,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,0.5,0.5,...,1.0,1.0,1.0,1.0,0.5,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.5,1.0,0.5,1.0,0.5,0.5,1.0,0.5


In [11]:
key_inbreds_status = status_df[status_df['key'] == 'key_inbreds'].iloc[0]['status']

if key_inbreds_status == 'ok':
    key_inbreds_path = DATA_DIR / EXPECTED_FILES['key_inbreds']
    key_inbreds = pd.read_csv(key_inbreds_path, sep='\t')
    print(f"key_inbreds shape: {key_inbreds.shape}")
    print(f"columns: {list(key_inbreds.columns)}")
    print(key_inbreds.head(10))
    print()
    print(f"Dataset value counts: {key_inbreds['Dataset'].value_counts().to_dict()}")
else:
    print(f"[skip] key_inbreds: {key_inbreds_status}")

key_inbreds shape: (2660, 7)
columns: ['Cultivar', 'Dataset', 'SourceName', 'Bioproject', 'BioSample', 'Alternative name', 'Comments']
  Cultivar   Dataset SourceName  Bioproject       BioSample Alternative name  \
0      B73  Assembly        NAM  PRJEB32225    SAMEA5569141              NaN   
1      B97  Assembly        NAM  PRJEB31061    SAMEA5313298              NaN   
2    CG108  Assembly        G2F  PRJEB59044  SAMEA112904665              NaN   
3    CG119  Assembly        G2F  PRJEB59044  SAMEA112904668              NaN   
4     CG44  Assembly        G2F  PRJEB59044  SAMEA112904666              NaN   
5   CML103  Assembly        NAM  PRJEB31061    SAMEA5313301              NaN   
6   CML228  Assembly        NAM  PRJEB31061    SAMEA5313302              NaN   
7   CML247  Assembly        NAM  PRJEB31061    SAMEA5313303              NaN   
8   CML322  Assembly        NAM  PRJEB31061    SAMEA5313305              NaN   
9   CML333  Assembly        NAM  PRJEB31061    SAMEA5313306      

## Testing data (2024 held-out set)
Same checks as training, applied to `Testing_data/`. No genotype file here --
the training genotype VCF/numerical matrix already covers 2014-2025 hybrids
and is shared across both splits. Two things specific to the test set matter
more than the individual file profiles: whether test environments are truly
disjoint from training environments (confirms this is a genuine year-based
holdout, not something that needs a custom split), and whether every hybrid
we need to predict actually has genotype coverage.

In [12]:
TEST_DATA_DIR = BASE_PATH / 'data' / 'raw' / 'Testing_data'

EXPECTED_TEST_FILES = {
    'submission_template': '1_Submission_Template_2024.csv',
    'test_meta':           '2_Testing_Meta_Data_2024.csv',
    'test_soil':           '3_Testing_Soil_Data_2024.csv',
    'test_weather_full':   '4_Testing_Weather_Data_2024_full_year.csv',
    'test_weather_seasons':'4_Testing_Weather_Data_2024_seasons_only.csv',
    'test_ec':             '6_Testing_EC_Data_2024.csv',
    'test_observed':       '7_Testing_Observed_Values.csv',
}

test_status_rows = []
for key, filename in EXPECTED_TEST_FILES.items():
    path = TEST_DATA_DIR / filename
    status = integrity_status(path)
    size = path.stat().st_size if path.exists() else None
    test_status_rows.append({'key': key, 'file': filename, 'status': status, 'size_bytes': size})

test_status_df = pd.DataFrame(test_status_rows)
print(test_status_df.to_string(index=False))

bad_test = test_status_df[test_status_df['status'] != 'ok']
if len(bad_test):
    print(f"\n{len(bad_test)} test file(s) failed the integrity check.")
else:
    print("\nAll test files passed the integrity check.")

                 key                                         file status  size_bytes
 submission_template               1_Submission_Template_2024.csv     ok      336283
           test_meta                 2_Testing_Meta_Data_2024.csv     ok       10888
           test_soil                 3_Testing_Soil_Data_2024.csv     ok        3697
   test_weather_full    4_Testing_Weather_Data_2024_full_year.csv     ok      702794
test_weather_seasons 4_Testing_Weather_Data_2024_seasons_only.csv     ok      424779
             test_ec                   6_Testing_EC_Data_2024.csv     ok      189448
       test_observed                7_Testing_Observed_Values.csv     ok      420463

All test files passed the integrity check.


In [13]:
test_tabular_frames: dict[str, pd.DataFrame] = {}

for key, filename in EXPECTED_TEST_FILES.items():
    row = test_status_df[test_status_df['key'] == key].iloc[0]
    if row['status'] != 'ok':
        print(f"[skip] {key} ({filename}): {row['status']}")
        continue

    df_full = pd.read_csv(TEST_DATA_DIR / filename)
    test_tabular_frames[key] = df_full

    print(f"=== {key} ({filename}) ===")
    print(f"shape: {df_full.shape}")
    print(f"columns: {list(df_full.columns)}")
    print(df_full.head(3))
    print()

=== submission_template (1_Submission_Template_2024.csv) ===
shape: (10057, 3)
columns: ['Env', 'Hybrid', 'Yield_Mg_ha']
         Env        Hybrid  Yield_Mg_ha
0  DEH1_2024  01CSI6/LH287          NaN
1  DEH1_2024  01DIB2/LH287          NaN
2  DEH1_2024  01DIB2/PHP02          NaN

=== test_meta (2_Testing_Meta_Data_2024.csv) ===
shape: (23, 40)
columns: ['Year', 'Env', 'Experiment_Code', 'Treatment', 'City', 'Farm', 'Field', 'Trial_ID (Assigned by collaborator for internal reference)', 'Soil_Taxonomic_ID and horizon description, if known', 'Weather_Station_Serial_Number (Last four digits, e.g. m2700s#####)', 'Weather_Station_Latitude (in decimal numbers NOT DMS)', 'Weather_Station_Longitude (in decimal numbers NOT DMS)', 'Date_weather_station_placed', 'Date_weather_station_removed', 'Previous_Crop', 'Pre-plant_tillage_method(s)', 'In-season_tillage_method(s)', 'Type_of_planter (fluted cone; belt cone; air planter)', 'System_Determining_Moisture', 'Pounds_Needed_Soil_Moisture', 'Latitud

In [14]:
for key, df_full in test_tabular_frames.items():
    miss = (df_full.isna().mean() * 100).round(1)
    miss = miss[miss > 0].sort_values(ascending=False)
    print(f"=== {key}: missing % by column ===")
    print(miss.to_string() if len(miss) else "(no missing values)")
    print()

=== submission_template: missing % by column ===
Yield_Mg_ha    100.0

=== test_meta: missing % by column ===
Issue/comment_#5                                                      100.0
Issue/comment_#7                                                      100.0
Issue/comment_#8                                                      100.0
Issue/comment_#6                                                      100.0
Issue/comment_#3                                                       91.3
Issue/comment_#4                                                       91.3
In-season_tillage_method(s)                                            73.9
Issue/comment_#2                                                       73.9
Soil_Taxonomic_ID and horizon description, if known                    65.2
Issue/comment_#1                                                       47.8
Date_weather_station_removed                                           39.1
Field                                                 

### Train/test environment disjointness
If this is a genuine year-based holdout, test environments should not
appear in the training environment sets at all -- any overlap means 2024
environments were also tested in earlier years, which changes what
"held-out" actually means here.

In [15]:
test_env_value_sets: dict[str, set] = {}

for key, df_full in test_tabular_frames.items():
    env_cols = [c for c in df_full.columns if 'env' in c.lower()]
    if not env_cols:
        print(f"{key}: no column with 'env' in its name -- columns are {list(df_full.columns)}")
        continue
    col = env_cols[0]
    test_env_value_sets[key] = set(df_full[col].astype(str).unique())
    print(f"{key}: using column '{col}' ({df_full[col].nunique()} unique values)")

print()
all_test_envs = set().union(*test_env_value_sets.values()) if test_env_value_sets else set()
all_train_envs = set().union(*env_value_sets.values()) if env_value_sets else set()
overlap = all_test_envs & all_train_envs
print(f"Test envs: {len(all_test_envs)} | Train envs: {len(all_train_envs)} | Overlap: {len(overlap)}")
if overlap:
    print(f"  OVERLAPPING envs (investigate before treating this as a clean holdout): {sorted(overlap)[:10]}")
else:
    print("  No overlap -- test environments are a genuine year-based holdout.")

submission_template: using column 'Env' (23 unique values)
test_meta: using column 'Env' (23 unique values)
test_soil: using column 'Env' (16 unique values)
test_weather_full: using column 'Env' (23 unique values)
test_weather_seasons: using column 'Env' (23 unique values)
test_ec: using column 'Env' (22 unique values)
test_observed: using column 'Env' (22 unique values)

Test envs: 23 | Train envs: 272 | Overlap: 0
  No overlap -- test environments are a genuine year-based holdout.


### Hybrid coverage check
Every hybrid in the submission template / observed values needs a row in
the genotype matrix for a genotype-alone model to produce a vote for it.

In [16]:
hybrid_check_key = 'test_observed' if 'test_observed' in test_tabular_frames else 'submission_template'

if hybrid_check_key in test_tabular_frames and 'geno_num' in dir():
    df_check = test_tabular_frames[hybrid_check_key]
    hybrid_cols = [c for c in df_check.columns if 'hybrid' in c.lower()]
    if hybrid_cols:
        col = hybrid_cols[0]
        test_hybrids = set(df_check[col].astype(str).unique())
        genotyped_hybrids = set(geno_num.index.astype(str))
        missing = test_hybrids - genotyped_hybrids
        print(f"{hybrid_check_key}: {len(test_hybrids)} unique hybrids (column '{col}')")
        print(f"Genotyped hybrids available: {len(genotyped_hybrids)}")
        print(f"Test hybrids WITHOUT genotype coverage: {len(missing)}")
        if missing:
            print(f"  sample missing: {sorted(missing)[:10]}")
    else:
        print(f"{hybrid_check_key}: no column with 'hybrid' in its name -- columns are {list(df_check.columns)}")
else:
    print("Skipped -- either the test frame or the genotype matrix (geno_num) isn't available above.")

test_observed: 1063 unique hybrids (column 'Hybrid')
Genotyped hybrids available: 5899
Test hybrids WITHOUT genotype coverage: 0


## Summary
Consolidated status -- what's confirmed usable vs. what still needs a
re-download or a readme check, so the next session knows exactly where to
pick up.

In [17]:
print("=== Training ===")
print(status_df.to_string(index=False))
print()
print("=== Testing ===")
print(test_status_df.to_string(index=False))
print()
print(f"Training tabular files loaded: {list(tabular_frames.keys())}")
print(f"Testing tabular files loaded: {list(test_tabular_frames.keys())}")
print(f"Env join-key sets compared (train): {list(env_value_sets.keys())}")
print(f"Env join-key sets compared (test): {list(test_env_value_sets.keys())}")

=== Training ===
             key                                                file status  size_bytes
           trait                 1_Training_Trait_Data_2014_2023.csv     ok    31590253
            meta                  2_Training_Meta_Data_2014_2023.csv     ok      100054
            soil                  3_Training_Soil_Data_2015_2023.csv     ok       28101
    weather_full     4_Training_Weather_Data_2014_2023_full_year.csv     ok    10460386
 weather_seasons  4_Training_Weather_Data_2014_2023_seasons_only.csv     ok     5493581
    genotype_vcf           5_Genotype_Data_All_2014_2025_Hybrids.vcf     ok    57432657
genotype_numeric 5_Genotype_Data_All_2014_2025_Hybrids_numerical.txt     ok    40765753
              ec                    6_Training_EC_Data_2014_2023.csv     ok     1971244
     key_inbreds                       key_inbreds_G2F_2014-2025.txt     ok      230060

=== Testing ===
                 key                                         file status  size_bytes
 

In [42]:
# Submission template vs. test_observed environment diagnostic
# Isolates the exact environment(s) present in one file but not the other,
# rather than just counting them.

def env_values(df: pd.DataFrame) -> set[str]:
    """Extracts the set of environment values from the first env-like column."""
    env_cols = [c for c in df.columns if 'env' in c.lower()]
    if not env_cols:
        raise ValueError(f"No env-like column found -- columns are {list(df.columns)}")
    return set(df[env_cols[0]].astype(str).str.strip().unique())


template_envs = env_values(test_tabular_frames['submission_template'])
observed_envs = env_values(test_tabular_frames['test_observed'])

print(f"submission_template: {len(template_envs)} unique environments")
print(f"test_observed:       {len(observed_envs)} unique environments")

only_in_template = template_envs - observed_envs
only_in_observed = observed_envs - template_envs

print(f"\nOnly in submission_template ({len(only_in_template)}): {sorted(only_in_template)}")
print(f"Only in test_observed ({len(only_in_observed)}): {sorted(only_in_observed)}")

if not only_in_template and not only_in_observed:
    print("\nSets are identical -- the earlier count mismatch was likely a "
          "duplicate row or NaN inflating one count, not a real env difference.")

submission_template: 23 unique environments
test_observed:       22 unique environments

Only in submission_template (1): ['SCH1_2024']
Only in test_observed (0): []


In [43]:
# Row-count sanity check for the flagged environment(s)
# If an environment is only in submission_template, check whether
# test_observed has zero rows for it, or a partial drop.

for env in sorted(only_in_template):
    sub_rows = test_tabular_frames['submission_template']
    env_col = [c for c in sub_rows.columns if 'env' in c.lower()][0]
    n_template_rows = (sub_rows[env_col].astype(str).str.strip() == env).sum()
    print(f"{env}: {n_template_rows} rows in submission_template, 0 in test_observed")

for env in sorted(only_in_observed):
    obs_rows = test_tabular_frames['test_observed']
    env_col = [c for c in obs_rows.columns if 'env' in c.lower()][0]
    n_observed_rows = (obs_rows[env_col].astype(str).str.strip() == env).sum()
    print(f"{env}: {n_observed_rows} rows in test_observed, 0 in submission_template")

SCH1_2024: 385 rows in submission_template, 0 in test_observed
